# Run SQL folder

Generic, reusable runner: executes every `*.sql` file of one folder in name order, one statement at a time.
Placeholders `${name}` are replaced with the job/task parameter of the same name; an unknown placeholder stops the run before anything executes.

Control parameters: `sql_folder` (path relative to this notebook, e.g. `../sql/20_base_views`) and `enabled` (`true` / `false`). Every other parameter is a placeholder value.

In [0]:
import os
import re

dbutils.widgets.text("sql_folder", "")
dbutils.widgets.text("enabled", "true")

CONTROL_KEYS = {"sql_folder", "enabled"}
params = dict(dbutils.widgets.getAll())
sql_folder = params.get("sql_folder", "").strip()
enabled = params.get("enabled", "true").strip().lower() == "true"
sql_vars = {k: v for k, v in params.items() if k not in CONTROL_KEYS}

if not sql_folder:
    raise ValueError("Parameter 'sql_folder' is required")

print(f"sql_folder = {sql_folder}\nenabled    = {enabled}")
for k in sorted(sql_vars):
    print(f"  ${{{k}}} = {sql_vars[k]}")

In [0]:
if not enabled:
    dbutils.notebook.exit(f"SKIPPED: {sql_folder} (enabled=false)")

In [0]:
PLACEHOLDER = re.compile(r"\$\{([A-Za-z_][A-Za-z0-9_]*)\}")


def split_statements(sql: str) -> list:
    """Split on ';' that is outside quotes, backticks and comments."""
    out, buf, i, n = [], [], 0, len(sql)
    quote = None
    while i < n:
        c = sql[i]
        nxt = sql[i + 1] if i + 1 < n else ""
        if quote:
            buf.append(c)
            if c == "\\" and quote in "'\"" and nxt:
                buf.append(nxt)
                i += 2
                continue
            if c == quote:
                quote = None
        elif c == "-" and nxt == "-":
            j = sql.find("\n", i)
            j = n if j == -1 else j
            buf.append(sql[i:j])
            i = j
            continue
        elif c == "/" and nxt == "*":
            j = sql.find("*/", i + 2)
            j = n if j == -1 else j + 2
            buf.append(sql[i:j])
            i = j
            continue
        elif c in "'\"`":
            quote = c
            buf.append(c)
        elif c == ";":
            out.append("".join(buf))
            buf = []
        else:
            buf.append(c)
        i += 1
    out.append("".join(buf))

    def has_code(s):
        no_comments = re.sub(r"--[^\n]*", "", s)
        no_comments = re.sub(r"/\*.*?\*/", "", no_comments, flags=re.S)
        return no_comments.strip() != ""

    return [s.strip() for s in out if has_code(s)]


def render(sql: str, file_name: str) -> str:
    missing = sorted({m for m in PLACEHOLDER.findall(sql) if m not in sql_vars})
    if missing:
        raise ValueError(f"{file_name}: no parameter for placeholder(s) {missing}")
    return PLACEHOLDER.sub(lambda m: sql_vars[m.group(1)], sql)

In [0]:
nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
nb_dir = os.path.dirname(nb_path if nb_path.startswith("/Workspace") else "/Workspace" + nb_path)
folder = os.path.normpath(os.path.join(nb_dir, sql_folder))
if not os.path.isdir(folder):
    raise FileNotFoundError(f"SQL folder not found: {folder}")

files = sorted(f for f in os.listdir(folder) if f.endswith(".sql"))
if not files:
    raise FileNotFoundError(f"No .sql files in {folder}")

plan = []
for f in files:
    with open(os.path.join(folder, f), encoding="utf-8") as fh:
        for idx, stmt in enumerate(split_statements(fh.read()), start=1):
            plan.append((f, idx, render(stmt, f)))

print(f"{len(files)} file(s), {len(plan)} statement(s) in {folder}")

In [0]:
for f, idx, stmt in plan:
    first_line = next((l for l in stmt.splitlines() if l.strip() and not l.strip().startswith("--")), "")
    print(f"-> {f} #{idx}: {first_line[:120]}")
    try:
        rows = spark.sql(stmt).limit(1).collect()
    except Exception as e:
        raise RuntimeError(f"{f} statement #{idx} failed: {e}") from e
    if rows and stmt.lstrip().upper().startswith("SELECT"):
        print(f"   result: {rows[0][0]}")

dbutils.notebook.exit(f"OK: {len(plan)} statement(s) from {len(files)} file(s) in {sql_folder}")